# 05 — CairoGo Data Leakage Analysis

**Owner:** Bassant Tarek (Person C — ID: 20231037)  
**Phase:** Phase 1 — Task 1c: Target Leakage Audit  
**Target Variable:** `request_failed` (`0` = completed, `1` = failed)  
**Prediction Point:** At ride request creation (`request_ts`)  
**Dataset:** `data/raw/cairogo_pack/` (`trips.csv`, `customers.csv`, `drivers.csv`, `zones.csv`)

---
## 1. Data Loading and Initial Setup

In this section, we import the required Python libraries and load the four CairoGo datasets: `trips`, `customers`, `drivers`, and `zones`.

The `request_ts`, `signup_date`, and `join_date` columns are parsed as datetime values to support temporal analysis during the data leakage investigation. We also verify that the dataset folder exists and inspect the shape of each dataset.

In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

print("Libraries loaded")

PACK = Path("../data/raw/cairogo_pack").resolve()
if not PACK.exists():
    raise FileNotFoundError(f"Dataset folder not found: {PACK}")

trips = pd.read_csv(PACK / "trips.csv", parse_dates=["request_ts"])
customers = pd.read_csv(PACK / "customers.csv", parse_dates=["signup_date"])
drivers = pd.read_csv(PACK / "drivers.csv", parse_dates=["join_date"])
zones = pd.read_csv(PACK / "zones.csv")

print(f"Datasets loaded successfully from: {PACK}")
print("\nDataset Shapes:")
print("Trips    :", trips.shape)
print("Customers:", customers.shape)
print("Drivers  :", drivers.shape)
print("Zones    :", zones.shape)

Libraries loaded
Datasets loaded successfully from: C:\Users\H P\cairogo-ride-failure-prediction\data\raw\cairogo_pack

Dataset Shapes:
Trips    : (27226, 18)
Customers: (6000, 5)
Drivers  : (1500, 6)
Zones    : (24, 5)


---
## 2. Dataset Schema Inspection

In this section, we inspect the column names of all four datasets to understand their structure and identify which columns may be known at request time versus which require further leakage investigation.

In [23]:
datasets = {
    "trips": trips,
    "customers": customers,
    "drivers": drivers,
    "zones": zones,
}

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"{name.upper()} COLUMNS")
    print("=" * 50)
    for i, col in enumerate(df.columns, start=1):
        print(f"{i}. {col}")


TRIPS COLUMNS
1. trip_id
2. customer_id
3. driver_id
4. request_ts
5. pickup_zone
6. dropoff_zone
7. distance_km
8. surge_multiplier
9. est_fare
10. eta_estimate_min
11. weather
12. payment_method
13. service_level
14. request_failed
15. actual_trip_time_min
16. cancel_reason
17. support_ticket_flag
18. driver_arrival_min

CUSTOMERS COLUMNS
1. customer_id
2. signup_date
3. channel
4. home_zone
5. loyalty_tier

DRIVERS COLUMNS
1. driver_id
2. join_date
3. vehicle_type
4. experience_months
5. home_zone
6. rating_avg

ZONES COLUMNS
1. zone_id
2. zone_name
3. legacy_name
4. zone_type
5. has_metro


---
## 3. Trips Dataset Inspection

Examines the main `trips` table:
- **Dataset Information:** Column data types and non-null counts.
- **First 10 Rows:** Sample structure and values.
- **Target Distribution & Missingness:** Counts of completed (`0`) and failed (`1`) requests.

In [24]:
print("Trips dataset information:")
trips.info()

print("\nFirst 10 rows:")
display(trips.head(10))

print("\nTarget column:")
print(trips["request_failed"].value_counts(dropna=False))

print("\nTarget missing values:", trips["request_failed"].isna().sum())

Trips dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 27226 entries, 0 to 27225
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   trip_id               27226 non-null  int64         
 1   customer_id           27226 non-null  int64         
 2   driver_id             27226 non-null  int64         
 3   request_ts            27226 non-null  datetime64[us]
 4   pickup_zone           27226 non-null  str           
 5   dropoff_zone          27226 non-null  str           
 6   distance_km           27226 non-null  float64       
 7   surge_multiplier      27226 non-null  float64       
 8   est_fare              27226 non-null  float64       
 9   eta_estimate_min      27226 non-null  float64       
 10  weather               25913 non-null  str           
 11  payment_method        27226 non-null  str           
 12  service_level         27226 non-null  str           
 13  

,trip_id,customer_id,driver_id,request_ts,pickup_zone,dropoff_zone,distance_km,surge_multiplier,est_fare,eta_estimate_min,weather,payment_method,service_level,request_failed,actual_trip_time_min,cancel_reason,support_ticket_flag,driver_arrival_min
0,100001,21924,5062,2026-05-01 00:01:46,Dokki,Badr City,17.22,1.1,107.4,6.4,NaN,card,economy,0,29.9,NaN,0,5.1
1,100002,20391,5681,2026-05-01 00:37:36,Dokki,Obour,12.74,1.1,114.9,7.3,rain,cash,comfort,0,28.9,NaN,0,7.0
2,100003,24083,5407,2026-05-01 00:40:12,Maadi,Obour,10.80,1.0,92.3,6.4,NaN,card,comfort,0,22.5,NaN,0,4.0
3,100004,25342,5833,2026-05-01 00:49:23,Dokki,Mohandessin,9.51,1.2,75.5,8.1,rain,cash,economy,0,28.9,NaN,0,6.1
4,100005,25697,6119,2026-05-01 01:05:33,Dokki,Ramses,4.54,1.0,39.3,8.6,rain,cash,economy,0,20.6,NaN,0,9.7
5,100006,24471,6296,2026-05-01 01:26:18,Ramses,Ramses,4.91,1.0,44.5,8.8,NaN,cash,economy,0,28.7,NaN,0,7.6
6,100007,20066,5267,2026-05-01 01:26:54,Abdeen,Abdeen,5.06,1.0,45.2,9.4,rain,cash,economy,0,25.7,NaN,0,10.7
7,100008,21953,6016,2026-05-01 01:34:15,Tora,Maadi,17.52,1.1,104.1,7.9,NaN,wallet,economy,1,NaN,driver_rejected,0,NaN
8,100009,21953,6043,2026-05-01 01:36:28,Tora,Maadi,17.52,1.0,94.6,4.9,NaN,card,economy,1,NaN,no_driver_found,0,NaN
9,100010,25023,5874,2026-05-01 01:52:33,Nasr City,Dokki,5.80,1.0,45.1,8.9,rain,wallet,economy,0,22.6,NaN,0,7.5



Target column:
request_failed
0    21560
1     5666
Name: count, dtype: int64

Target missing values: 0


---
## 4. Missing Values Analysis by Target Class

Compares the percentage of missing values in each column between completed (`0`) and failed (`1`) ride requests (`Difference = Failed (1) Missing % - Completed (0) Missing %`). Large class-conditioned missingness gaps indicate post-outcome columns.

In [25]:
missing_comparison = trips.groupby("request_failed").apply(
    lambda group: group.isna().mean() * 100,
    include_groups=False,
).T

missing_comparison.columns = ["Completed (0)", "Failed (1)"]
missing_comparison["Difference"] = (
    missing_comparison["Failed (1)"] - missing_comparison["Completed (0)"]
)

display(missing_comparison.round(2))

,Completed (0),Failed (1),Difference
trip_id,0.00,0.00,0.00
customer_id,0.00,0.00,0.00
driver_id,0.00,0.00,0.00
request_ts,0.00,0.00,0.00
pickup_zone,0.00,0.00,0.00
dropoff_zone,0.00,0.00,0.00
distance_km,0.00,0.00,0.00
surge_multiplier,0.00,0.00,0.00
est_fare,0.00,0.00,0.00
eta_estimate_min,0.00,0.00,0.00


---
## 5. Identification of Confirmed Data Leakage Features

Based on the missingness-by-target audit, three columns are confirmed post-request leakage:
- **`actual_trip_time_min`:** Actual trip duration (100% missing for failed requests), known only after completion.
- **`cancel_reason`:** Cancellation/failure reason (100% missing for completed requests), directly revealing outcome.
- **`driver_arrival_min`:** Actual driver arrival time (100% missing for failed requests), unavailable at request creation.

In [26]:
confirmed_leakage = [
    "actual_trip_time_min",
    "cancel_reason",
    "driver_arrival_min",
]

print("Confirmed Leakage Columns:")
for col in confirmed_leakage:
    print(f"- {col}")

print("\nThese columns must be excluded from model training.")

Confirmed Leakage Columns:
- actual_trip_time_min
- cancel_reason
- driver_arrival_min

These columns must be excluded from model training.


---
## 6. Investigation of Suspicious & Profile Features

### 6.1 Suspicious Trip Columns (`driver_id` and `support_ticket_flag`)
Although `driver_id` and `support_ticket_flag` have no missing values, their availability at prediction time (`request_ts`) must be verified.

In [27]:
suspicious_columns = ["driver_id", "support_ticket_flag"]

for col in suspicious_columns:
    print(f"\n{'=' * 50}")
    print(f"Column: {col}")
    print("=" * 50)

    print("\nValue counts by target:")
    display(pd.crosstab(trips[col], trips["request_failed"], margins=True))

    print("\nFailure rate by value:")
    display(
        trips.groupby(col)["request_failed"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "failure_rate"})
        .round(3)
    )


Column: driver_id

Value counts by target:


request_failed,0,1,All
driver_id,,,
5000,17,5,22
5001,28,7,35
5002,9,2,11
5003,26,6,32
5004,22,6,28
...,...,...,...
6496,14,4,18
6497,39,4,43
6498,16,1,17



Failure rate by value:


,count,failure_rate
driver_id,,
5000,22,0.227
5001,35,0.200
5002,11,0.182
5003,32,0.188
5004,28,0.214
...,...,...
6495,17,0.294
6496,18,0.222
6497,43,0.093



Column: support_ticket_flag

Value counts by target:


request_failed,0,1,All
support_ticket_flag,,,
0,21132,4275,25407
1,428,1391,1819
All,21560,5666,27226



Failure rate by value:


,count,failure_rate
support_ticket_flag,,
0,25407,0.168
1,1819,0.765


#### 6.1 Findings & Decisions

- **`driver_id` — Decision: EXCLUDE UNTIL VERIFIED.** Failure rates vary substantially across drivers, but variation alone does not prove leakage. Because the dataset lacks a driver-assignment timestamp, we cannot assume `driver_id` is known when the request is created.
- **`support_ticket_flag` — Decision: POTENTIAL TEMPORAL LEAKAGE (EXCLUDE).** Requests with `support_ticket_flag = 1` have a 76.5% failure rate vs. 16.8% when `0`. Support tickets are typically created after a failure or issue occurs.

### 6.2 Driver Join-Date Consistency Check
Compares each driver's `join_date` with the corresponding trip's `request_ts`.

In [28]:
driver_check = trips[
    ["trip_id", "driver_id", "request_ts", "request_failed"]
].merge(drivers[["driver_id", "join_date"]], on="driver_id", how="left")

driver_check["joined_after_request"] = (
    driver_check["join_date"] > driver_check["request_ts"]
)

print("Driver join-date consistency check:")
print(driver_check["joined_after_request"].value_counts(dropna=False))

print("\nJoined after request by target:")
display(
    pd.crosstab(
        driver_check["joined_after_request"],
        driver_check["request_failed"],
        margins=True,
    )
)

Driver join-date consistency check:
joined_after_request
False    27226
Name: count, dtype: int64

Joined after request by target:


request_failed,0,1,All
joined_after_request,,,
False,21560,5666,27226
All,21560,5666,27226


#### 6.2 Results & Decision
- Total requests checked: 27,226 | Drivers who joined after the request date: **0**.
- **Conclusion:** All drivers joined before the corresponding requests. However, this does not prove `driver_id` was assigned at request creation — retain `driver_id` only if assignment at request time is confirmed.

### 6.3 Customer Signup-Date Consistency Check
Compares each customer's `signup_date` with the corresponding trip's `request_ts`.

In [29]:
customer_check = trips[
    ["trip_id", "customer_id", "request_ts", "request_failed"]
].merge(customers[["customer_id", "signup_date"]], on="customer_id", how="left")

customer_check["signup_after_request"] = (
    customer_check["signup_date"] > customer_check["request_ts"]
)

print("Customer signup-date consistency:")
print(customer_check["signup_after_request"].value_counts(dropna=False))

print("\nSignup after request by target:")
display(
    pd.crosstab(
        customer_check["signup_after_request"],
        customer_check["request_failed"],
        margins=True,
    )
)

Customer signup-date consistency:
signup_after_request
False    27226
Name: count, dtype: int64

Signup after request by target:


request_failed,0,1,All
signup_after_request,,,
False,21560,5666,27226
All,21560,5666,27226


#### 6.3 Results & Decision
- Total requests checked: 27,226 | Customers who signed up after the request: **0**.
- **Decision:** `signup_date` is temporally consistent with prediction time. Mutable customer attributes such as `loyalty_tier` still require historical snapshot verification.

### 6.4 Customer and Driver Feature Inspection
Summarizes distributions and categorical values for `customers` and `drivers` profile attributes.

In [30]:
print("=" * 50)
print("CUSTOMER FEATURES")
print("=" * 50)
display(customers.describe(include="all").T)

for col in ["channel", "home_zone", "loyalty_tier"]:
    print(f"\n{col}:")
    print(customers[col].value_counts(dropna=False))

print("\n" + "=" * 50)
print("DRIVER FEATURES")
print("=" * 50)
display(drivers.describe(include="all").T)

for col in ["vehicle_type", "home_zone"]:
    print(f"\n{col}:")
    print(drivers[col].value_counts(dropna=False))

CUSTOMER FEATURES


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_id,6000.0,NaN,NaN,NaN,22999.5,20000.0,21499.75,22999.5,24499.25,25999.0,1732.195139
signup_date,6000,NaN,NaN,NaN,2026-03-05 05:33:36,2025-10-14 00:00:00,2026-01-08 00:00:00,2026-03-05 00:00:00,2026-04-25 00:00:00,2026-09-24 00:00:00,NaN
channel,6000,3,organic,2986,NaN,NaN,NaN,NaN,NaN,NaN,NaN
home_zone,6000,24,Nasr City,437,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loyalty_tier,6000,3,bronze,3000,NaN,NaN,NaN,NaN,NaN,NaN,NaN



channel:
channel
organic     2986
ads         2119
referral     895
Name: count, dtype: int64

home_zone:
home_zone
Nasr City        437
Dokki            401
Shubra           399
Heliopolis       364
Mohandessin      350
Maadi            337
Abdeen           306
Agouza           298
Garden City      276
Shorouk          273
Ramses           256
Zamalek          247
Downtown         240
Cairo Airport    235
Obour            208
New Cairo        204
Madinaty         193
6th October      191
Qalyub           183
Sheikh Zayed     144
Helwan           133
Khanka           112
Badr City        108
Tora             105
Name: count, dtype: int64

loyalty_tier:
loyalty_tier
bronze    3000
silver    1800
gold      1200
Name: count, dtype: int64

DRIVER FEATURES


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
driver_id,1500.0,NaN,NaN,NaN,5749.5,5000.0,5374.75,5749.5,6124.25,6499.0,433.157015
join_date,1500,NaN,NaN,NaN,2026-03-16 14:21:07.200000,2025-12-17 00:00:00,2026-02-13 00:00:00,2026-03-11 00:00:00,2026-04-06 00:00:00,2026-08-17 00:00:00,NaN
vehicle_type,1500,3,sedan,871,NaN,NaN,NaN,NaN,NaN,NaN,NaN
experience_months,1500.0,NaN,NaN,NaN,38.52,1.0,19.0,32.0,52.0,140.0,26.365502
home_zone,1500,24,Nasr City,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating_avg,1238.0,NaN,NaN,NaN,4.546502,3.92,4.42,4.54,4.66,5.0,0.184513



vehicle_type:
vehicle_type
sedan        871
hatchback    458
van          171
Name: count, dtype: int64

home_zone:
home_zone
Nasr City        110
Heliopolis       107
Shubra           103
Mohandessin       87
Agouza            86
Zamalek           80
Ramses            79
Dokki             75
Maadi             67
Garden City       62
New Cairo         62
Downtown          59
Shorouk           58
Cairo Airport     57
Abdeen            57
6th October       55
Qalyub            52
Madinaty          47
Obour             42
Sheikh Zayed      40
Helwan            32
Badr City         30
Tora              30
Khanka            23
Name: count, dtype: int64


#### 6.4 Findings & Decisions
- **Customer features:** `signup_date` predates all requests; `channel` and `home_zone` are potentially usable if known at request time; `loyalty_tier` requires snapshot verification; `customer_id` is an identifier.
- **Driver features:** `join_date` predates all requests; `vehicle_type` and `home_zone` depend on driver assignment at request time; `experience_months` and `rating_avg` require reference-date / pre-request aggregation verification.

### 6.5 Customer and Driver Missing Values Analysis
Checks for missing values across `customers` and `drivers` profile tables.

In [31]:
print("CUSTOMER FEATURE MISSING VALUES")
display(customers.isna().sum())

print("\nDRIVER FEATURE MISSING VALUES")
display(drivers.isna().sum())

CUSTOMER FEATURE MISSING VALUES


customer_id     0
signup_date     0
channel         0
home_zone       0
loyalty_tier    0
dtype: int64


DRIVER FEATURE MISSING VALUES


driver_id              0
join_date              0
vehicle_type           0
experience_months      0
home_zone              0
rating_avg           262
dtype: int64

#### 6.5 Findings
- All `customers` columns are complete (0 missing values).
- In `drivers`, `rating_avg` has 262 missing values out of 1,500 drivers (~17.47%), reflecting drivers without enough rated trips rather than direct outcome leakage.

### 6.6 Driver Experience Temporal Consistency
Compares each driver's recorded `experience_months` against their platform tenure at `request_ts`.

In [32]:
driver_temporal = trips[["trip_id", "driver_id", "request_ts"]].merge(
    drivers[["driver_id", "join_date", "experience_months"]],
    on="driver_id",
    how="left",
)
driver_temporal["platform_tenure_months"] = (
    (driver_temporal["request_ts"] - driver_temporal["join_date"]).dt.days / 30.44
)

print("Driver experience compared with platform tenure:")
display(driver_temporal[["experience_months", "platform_tenure_months"]].describe().round(2))

Driver experience compared with platform tenure:


,experience_months,platform_tenure_months
count,27226.00,27226.00
mean,37.97,3.69
std,25.97,1.56
min,1.00,0.00
25%,19.00,2.56
50%,32.00,3.71
75%,52.00,4.83
max,140.00,8.28


#### 6.6 Results & Decision
- Average `experience_months` (37.97) is much larger than average `platform_tenure_months` (3.69), which is consistent with prior driving experience before joining CairoGo.
- **Decision:** No obvious temporal inconsistency, though the reference date used to compute `experience_months` should be verified.

### 6.7 Loyalty Tier and Driver Rating Analysis
Examines target failure rates across `loyalty_tier` and `rating_avg` availability.

In [33]:
trips_with_profiles = (
    trips.merge(customers[["customer_id", "loyalty_tier"]], on="customer_id", how="left")
    .merge(drivers[["driver_id", "rating_avg"]], on="driver_id", how="left")
)

print("Loyalty tier by request outcome:")
display(pd.crosstab(trips_with_profiles["loyalty_tier"], trips_with_profiles["request_failed"], margins=True))

print("\nFailure rate by loyalty tier:")
display(
    trips_with_profiles.groupby("loyalty_tier")["request_failed"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "failure_rate"})
    .round(3)
)

print("\nRating missingness by target:")
display(pd.crosstab(trips_with_profiles["rating_avg"].isna(), trips_with_profiles["request_failed"], margins=True))

print("\nRating statistics by target:")
display(
    trips_with_profiles.groupby("request_failed")["rating_avg"]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)

Loyalty tier by request outcome:


request_failed,0,1,All
loyalty_tier,,,
bronze,3455,1625,5080
gold,11014,2157,13171
silver,7091,1884,8975
All,21560,5666,27226



Failure rate by loyalty tier:


,count,failure_rate
loyalty_tier,,
bronze,5080,0.320
gold,13171,0.164
silver,8975,0.210



Rating missingness by target:


request_failed,0,1,All
rating_avg,,,
False,19478,4946,24424
True,2082,720,2802
All,21560,5666,27226



Rating statistics by target:


,count,mean,min,max
request_failed,,,,
0,19478,4.559,3.92,5.0
1,4946,4.556,3.92,5.0


#### 6.7 Results & Decisions
- **Loyalty tier:** Failure rates differ across tiers (Bronze 32.0%, Silver 21.0%, Gold 16.4%). Association does not prove leakage, but `loyalty_tier` must reflect the customer's tier at `request_ts`.
- **Driver rating:** Mean ratings are nearly identical for completed (4.559) and failed (4.556) trips, whereas missing `rating_avg` exhibits a higher failure rate. Safe to use only if computed strictly from pre-request ratings.

### 6.8 Zones Dataset Inspection & Zone Matching
Inspects `zones.csv` and verifies that joining `pickup_zone` and `dropoff_zone` against both `zone_name` and `legacy_name` produces zero unmatched records or row inflation.

In [34]:
print("Zones dataset information:")
zones.info()

print("\nFirst 10 rows:")
display(zones.head(10))

print("\nZone type distribution:")
display(zones["zone_type"].value_counts(dropna=False))

print("\nMetro availability:")
display(zones["has_metro"].value_counts(dropna=False))

print("\nMissing values:")
display(zones.isnull().sum())

Zones dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   zone_id      24 non-null     str  
 1   zone_name    24 non-null     str  
 2   legacy_name  24 non-null     str  
 3   zone_type    24 non-null     str  
 4   has_metro    24 non-null     int64
dtypes: int64(1), str(4)
memory usage: 1.1 KB

First 10 rows:


,zone_id,zone_name,legacy_name,zone_type,has_metro
0,Z01,Downtown,Downtown,downtown,1
1,Z02,Zamalek,Zamalek,downtown,1
2,Z03,Garden City,Garden City,downtown,1
3,Z04,Abdeen,Abdeen,downtown,1
4,Z05,Shubra District,Shubra,downtown,1
5,Z06,Ramses,Ramses,downtown,1
6,Z07,Maadi,Maadi,midtown,1
7,Z08,Nasr City,Nasr City,midtown,1
8,Z09,Heliopolis,Heliopolis,midtown,1
9,Z10,Mohandessin,Mohandessin,midtown,0



Zone type distribution:


zone_type
downtown    6
midtown     6
new_city    6
suburb      5
airport     1
Name: count, dtype: int64


Metro availability:


has_metro
0    15
1     9
Name: count, dtype: int64


Missing values:


zone_id        0
zone_name      0
legacy_name    0
zone_type      0
has_metro      0
dtype: int64

In [35]:
zone_lookup = pd.concat(
    [
        zones[["zone_name", "zone_type", "has_metro"]].rename(columns={"zone_name": "zone_label"}),
        zones[["legacy_name", "zone_type", "has_metro"]].rename(columns={"legacy_name": "zone_label"}),
    ],
    ignore_index=True,
).drop_duplicates(subset=["zone_label"])

zone_lookup["zone_key"] = zone_lookup["zone_label"].astype(str).str.strip().str.casefold()

pickup_lookup = zone_lookup[["zone_key", "zone_type", "has_metro"]].rename(
    columns={"zone_type": "pickup_zone_type", "has_metro": "pickup_has_metro"}
)
dropoff_lookup = zone_lookup[["zone_key", "zone_type", "has_metro"]].rename(
    columns={"zone_type": "dropoff_zone_type", "has_metro": "dropoff_has_metro"}
)

trips_zones = trips.copy()
trips_zones["pickup_zone_key"] = trips_zones["pickup_zone"].astype(str).str.strip().str.casefold()
trips_zones = trips_zones.merge(
    pickup_lookup, left_on="pickup_zone_key", right_on="zone_key", how="left", validate="many_to_one"
).drop(columns=["pickup_zone_key", "zone_key"])

trips_zones["dropoff_zone_key"] = trips_zones["dropoff_zone"].astype(str).str.strip().str.casefold()
trips_zones = trips_zones.merge(
    dropoff_lookup, left_on="dropoff_zone_key", right_on="zone_key", how="left", validate="many_to_one"
).drop(columns=["dropoff_zone_key", "zone_key"])

unmatched_pickup = sorted(trips.loc[trips_zones["pickup_zone_type"].isna(), "pickup_zone"].dropna().unique())
unmatched_dropoff = sorted(trips.loc[trips_zones["dropoff_zone_type"].isna(), "dropoff_zone"].dropna().unique())

print("Trips shape:", trips.shape, "| Merged shape:", trips_zones.shape)
print("Missing pickup zone attributes :", trips_zones["pickup_zone_type"].isna().sum())
print("Missing dropoff zone attributes:", trips_zones["dropoff_zone_type"].isna().sum())
print("Unmatched pickup labels :", unmatched_pickup)
print("Unmatched dropoff labels:", unmatched_dropoff)

assert len(trips_zones) == len(trips), "Zone merge changed the number of trip rows."
assert len(unmatched_pickup) == 0 and len(unmatched_dropoff) == 0, "Unmatched zones remain."

Trips shape: (27226, 18) | Merged shape: (27226, 22)
Missing pickup zone attributes : 0
Missing dropoff zone attributes: 0
Unmatched pickup labels : []
Unmatched dropoff labels: []


#### 6.8 Assessment
The `zones` table contains 24 static geographic reference rows (`zone_type`, `has_metro`) with 0 missing values. No direct outcome leakage is present in zone attributes, provided both `zone_name` and `legacy_name` are handled during joins.

---
## 7. Additional Data Leakage & Integrity Checks

### 7.1 Target Integrity, Exact Duplicates & Primary Key Uniqueness
Verifies target label validity, exact duplicate rows, and identifier uniqueness.

In [36]:
print("Target values:")
display(trips["request_failed"].value_counts(dropna=False).sort_index())

print("\nTarget proportions (%):")
display((trips["request_failed"].value_counts(normalize=True) * 100).round(2))

print("Exact duplicate trip rows:", int(trips.duplicated().sum()))
print("trip_id unique    :", trips["trip_id"].is_unique)
print("customer_id unique:", customers["customer_id"].is_unique)
print("driver_id unique  :", drivers["driver_id"].is_unique)

assert trips["trip_id"].is_unique, "trip_id is not unique."
assert trips["request_failed"].notna().all(), "Target contains missing values."
assert set(trips["request_failed"].dropna().unique()).issubset({0, 1}), "Unexpected target labels found."

Target values:


request_failed
0    21560
1     5666
Name: count, dtype: int64


Target proportions (%):


request_failed
0    79.19
1    20.81
Name: proportion, dtype: float64

Exact duplicate trip rows: 0
trip_id unique    : True
customer_id unique: True
driver_id unique  : True


#### Interpreting the Duplicate Checks
- `trip_id`, `customer_id`, and `driver_id` are unique in their respective primary tables.
- A unique identifier is not automatically leakage, but identifiers should not be used as raw predictive features without justification.

### 7.2 Referential Integrity Checks
Verifies that every `customer_id` and `driver_id` in `trips` exists in `customers` and `drivers`.

In [37]:
missing_customer_profiles = int((~trips["customer_id"].isin(customers["customer_id"])).sum())
missing_driver_profiles = int((~trips["driver_id"].isin(drivers["driver_id"])).sum())

print("Trips without a matching customer profile:", missing_customer_profiles)
print("Trips without a matching driver profile  :", missing_driver_profiles)

assert missing_customer_profiles == 0 and missing_driver_profiles == 0

Trips without a matching customer profile: 0
Trips without a matching driver profile  : 0


---
## 8. Temporal Analysis and Potential Time Leakage

### 8.1 Timestamp Range & Time-Based Failure Patterns
Inspects timestamp coverage, chronological ordering, and daily/hourly failure patterns to inform a chronological train/validation split.

In [38]:
print("First request:", trips["request_ts"].min())
print("Last request :", trips["request_ts"].max())
print("Rows already sorted by request_ts:", trips["request_ts"].is_monotonic_increasing)

time_audit = trips[["request_ts", "request_failed"]].copy()
time_audit["request_date"] = time_audit["request_ts"].dt.date
time_audit["request_hour"] = time_audit["request_ts"].dt.hour
time_audit["request_dayofweek"] = time_audit["request_ts"].dt.day_name()

daily_failure = time_audit.groupby("request_date")["request_failed"].agg(requests="count", failure_rate="mean")
daily_failure["failure_rate_pct"] = daily_failure["failure_rate"] * 100

hourly_failure = time_audit.groupby("request_hour")["request_failed"].agg(requests="count", failure_rate="mean")
hourly_failure["failure_rate_pct"] = hourly_failure["failure_rate"] * 100

print("\nDaily failure-rate summary:")
display(daily_failure.describe().round(3))

print("\nHourly failure rates:")
display(hourly_failure.round(3))

First request: 2026-05-01 00:01:46
Last request : 2026-08-27 23:57:39
Rows already sorted by request_ts: True

Daily failure-rate summary:


,requests,failure_rate,failure_rate_pct
count,119.000,119.000,119.000
mean,228.790,0.208,20.765
std,25.419,0.033,3.349
min,170.000,0.126,12.560
25%,210.000,0.184,18.389
50%,230.000,0.209,20.939
75%,245.000,0.229,22.857
max,295.000,0.295,29.524



Hourly failure rates:


,requests,failure_rate,failure_rate_pct
request_hour,,,
0,867,0.306,30.565
1,765,0.282,28.235
2,593,0.261,26.138
3,471,0.278,27.813
4,338,0.249,24.852
5,369,0.268,26.829
6,765,0.299,29.935
7,1699,0.201,20.071
8,2197,0.077,7.692


### 8.2 Trip ID Temporal Ordering Check
Checks whether `trip_id` is monotonically ordered with `request_ts`. Temporal variation across hours/days reflects demand patterns, so splits must be chronological to avoid training on future requests.

In [39]:
id_time = trips[["trip_id", "request_ts"]].sort_values("trip_id")
print("Trip ID increases with request time:", id_time["request_ts"].is_monotonic_increasing)

print("\nFirst and last records after sorting by trip_id:")
display(pd.concat([id_time.head(3), id_time.tail(3)]))

Trip ID increases with request time: True

First and last records after sorting by trip_id:


,trip_id,request_ts
0,100001,2026-05-01 00:01:46
1,100002,2026-05-01 00:37:36
2,100003,2026-05-01 00:40:12
27223,127224,2026-08-27 23:48:19
27224,127225,2026-08-27 23:49:51
27225,127226,2026-08-27 23:57:39


---
## 9. Feature Audit Based on Prediction Time

Consolidates feature-level eligibility decisions assuming predictions are made at `request_ts`.

In [40]:
feature_decisions = pd.DataFrame([
    ("trip_id", "Identifier", "Exclude initially", "Technical identifier; may encode temporal ordering and is not a generalizable behavioral feature."),
    ("customer_id", "Identifier", "Exclude initially", "Customer identifier; do not use as a raw feature without justification."),
    ("driver_id", "Requires verification", "Conditional", "Verify that the driver had been assigned before prediction time; its presence in the row does not prove this."),
    ("request_ts", "Request timestamp", "Use derived features only", "Known at request time; hour or day features may be derived if appropriate."),
    ("pickup_zone", "Location", "Provisionally allowed", "Pickup location is generally available when the request is created."),
    ("dropoff_zone", "Location", "Provisionally allowed", "Destination is generally available when the request is created."),
    ("distance_km", "Route estimate", "Verify", "Confirm that this is an estimated distance available at request time, not an actual distance measured after the trip."),
    ("surge_multiplier", "Pricing", "Allowed if verified", "Must be the pricing value calculated at request time."),
    ("est_fare", "Fare estimate", "Allowed if verified", "Must be the estimated fare at request time, not the final fare."),
    ("eta_estimate_min", "Estimate", "Allowed if verified", "The ETA must be available at request time."),
    ("weather", "Context", "Allowed if verified", "Must represent weather at request time; also review missing values."),
    ("payment_method", "User selection", "Provisionally allowed", "Allowed if selected by the user before submitting the request."),
    ("service_level", "User selection", "Provisionally allowed", "Allowed if the service level is selected before prediction."),
    ("request_failed", "Target", "Never include in X", "This is the variable being predicted."),
    ("actual_trip_time_min", "Post-event", "Confirmed leakage", "Actual duration is not known when the request is created."),
    ("cancel_reason", "Outcome / failure reason", "Confirmed leakage", "Reveals the cancellation or failure reason after the event."),
    ("support_ticket_flag", "Likely post-event", "Exclude until verified", "Strongly associated with failure and may be created after the issue; the ticket timestamp must be checked."),
    ("driver_arrival_min", "Post-request", "Confirmed leakage", "Actual arrival time is not known when the request is created."),
    ("signup_date", "Customer profile", "Allowed if verified", "The signup date predates the request in the current check; use account age as of the request time."),
    ("channel", "Customer profile", "Allowed if verified", "Confirm that the customer acquisition channel is stable and known at request time."),
    ("home_zone", "Customer / driver profile", "Allowed if verified", "Verify that the recorded location is the version available at request time."),
    ("loyalty_tier", "Customer profile", "Requires verification", "May change over time; a historical snapshot as of request_ts is required."),
    ("join_date", "Driver profile", "Allowed if verified", "The join date predates the request in the current check; tenure can be derived as of the request time."),
    ("vehicle_type", "Driver profile", "Requires verification", "Verify that the vehicle type was known for the assigned driver at request time."),
    ("experience_months", "Driver profile", "Requires verification", "The reference date used to calculate experience must be known."),
    ("rating_avg", "Driver profile", "Requires verification", "The average must be calculated using ratings recorded before the request only."),
    ("zone_type / has_metro", "Geographic reference", "Provisionally allowed", "Relatively stable geographic attributes; verify zone-name matching."),
], columns=["feature", "category", "decision", "reason"])

display(feature_decisions)

,feature,category,decision,reason
0,trip_id,Identifier,Exclude initially,Technical identifier; may encode temporal orde...
1,customer_id,Identifier,Exclude initially,Customer identifier; do not use as a raw featu...
2,driver_id,Requires verification,Conditional,Verify that the driver had been assigned befor...
3,request_ts,Request timestamp,Use derived features only,Known at request time; hour or day features ma...
4,pickup_zone,Location,Provisionally allowed,Pickup location is generally available when th...
5,dropoff_zone,Location,Provisionally allowed,Destination is generally available when the re...
6,distance_km,Route estimate,Verify,Confirm that this is an estimated distance ava...
7,surge_multiplier,Pricing,Allowed if verified,Must be the pricing value calculated at reques...
8,est_fare,Fare estimate,Allowed if verified,"Must be the estimated fare at request time, no..."
9,eta_estimate_min,Estimate,Allowed if verified,The ETA must be available at request time.


---
## 10. Investigating Suspicious Feature Associations

Compares failure rates across categories for selected sensitive features to document signals that warrant creation-time verification.

In [41]:
association_data = trips.merge(
    customers[["customer_id", "loyalty_tier"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

association_columns = [
    col
    for col in ["support_ticket_flag", "payment_method", "service_level", "weather", "loyalty_tier"]
    if col in association_data.columns
]

for col in association_columns:
    print(f"\nFailure rate by {col}:")
    display(
        association_data.groupby(col, dropna=False)["request_failed"]
        .agg(requests="size", failure_rate="mean")
        .assign(failure_rate_pct=lambda x: x["failure_rate"] * 100)
        .round(3)
        .sort_values("failure_rate", ascending=False)
    )


Failure rate by support_ticket_flag:


,requests,failure_rate,failure_rate_pct
support_ticket_flag,,,
1,1819,0.765,76.471
0,25407,0.168,16.826



Failure rate by payment_method:


,requests,failure_rate,failure_rate_pct
payment_method,,,
cash,14228,0.221,22.083
card,10126,0.196,19.603
wallet,2872,0.188,18.767



Failure rate by service_level:


,requests,failure_rate,failure_rate_pct
service_level,,,
economy,19150,0.260,25.990
comfort,5926,0.101,10.074
premium,2150,0.043,4.279



Failure rate by weather:


,requests,failure_rate,failure_rate_pct
weather,,,
rain,1661,0.240,24.022
NaN,1313,0.239,23.915
dust,1858,0.223,22.282
clear,22394,0.203,20.269



Failure rate by loyalty_tier:


,requests,failure_rate,failure_rate_pct
loyalty_tier,,,
bronze,5080,0.320,31.988
silver,8975,0.210,20.992
gold,13171,0.164,16.377


---
## 11. Leakage Prevention Rules for Model Development

Apply the following rules during Modeling:
1. Separate `request_failed` from `X`.
2. Drop confirmed leakage columns (`actual_trip_time_min`, `cancel_reason`, `driver_arrival_min`) and exclude `support_ticket_flag` and `driver_id` until availability timestamps are verified.
3. Do not use post-request columns to impute missing values.
4. Fit imputation, encoding, and scaling inside a Pipeline on training data only.
5. Use a chronological train/validation split (`trips.csv` -> `recent_labelled.csv`).
6. Verify historical snapshot semantics before joining mutable profile attributes.

In [42]:
confirmed_leakage = ["actual_trip_time_min", "cancel_reason", "driver_arrival_min"]
exclude_until_verified = ["support_ticket_flag", "driver_id"]
identifier_columns = ["trip_id", "customer_id"]
target_column = "request_failed"

print("Confirmed leakage:", confirmed_leakage)
print("Exclude until timing is verified:", exclude_until_verified)
print("Identifiers not recommended as raw predictors:", identifier_columns)
print("Target:", target_column)

initial_excluded = set(
    confirmed_leakage + exclude_until_verified + identifier_columns + [target_column]
)
candidate_trip_features = [c for c in trips.columns if c not in initial_excluded]

print("\nCandidate trip-level columns (subject to availability checks):")
print(candidate_trip_features)

Confirmed leakage: ['actual_trip_time_min', 'cancel_reason', 'driver_arrival_min']
Exclude until timing is verified: ['support_ticket_flag', 'driver_id']
Identifiers not recommended as raw predictors: ['trip_id', 'customer_id']
Target: request_failed

Candidate trip-level columns (subject to availability checks):
['request_ts', 'pickup_zone', 'dropoff_zone', 'distance_km', 'surge_multiplier', 'est_fare', 'eta_estimate_min', 'weather', 'payment_method', 'service_level']


---
## 12. Final Data Leakage Audit Summary

### Confirmed Leakage
- `actual_trip_time_min`
- `cancel_reason`
- `driver_arrival_min`

### Exclude Until Verified
- `support_ticket_flag`: Failure rate jumps to 76.5% when `1`; ticket creation timestamp must be checked.
- `driver_id`: No driver-assignment timestamp is available; `join_date <= request_ts` does not prove assignment at request time.

### Features Requiring Historical Snapshot Verification
- `loyalty_tier`, `rating_avg`, `experience_months`, `vehicle_type`, and customer/driver `home_zone`.

### Additional Data Quality Notes
- Zone joins must recognize both `zone_name` and `legacy_name` to avoid join loss.
- **Final decision:** Exclude all post-`request_ts` information, keep unverified columns out until documented, and fit all preprocessing strictly on training data.